# Variational Autoencoder for Image Generation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EmoryMLIP/DeepGenerativeModelingIntro/blob/main/examples/VAE.ipynb)

## Some Reference

- Original paper by [Kingma and Welling (2013)](https://arxiv.org/abs/1312.6114) and [Rezende et al. (2014)](https://arxiv.org/abs/1401.4082)
- Comprehensive review by [Kingma and Welling (2019)](https://arxiv.org/abs/1906.02691)
- Introduction by [Ruthotto and Haber (2021)](https://arxiv.org/abs/2103.05180)
  - Our code is based on [theirs](https://github.com/EmoryMLIP/DeepGenerativeModelingIntro)
- Kian Ming A. Chai (Data61)'s Variational Learning of Fractional Posteriors (unpublished)

## Setup
### Key parameters

In [ ]:
### Data
datasetstr = "MNIST" # MNIST or FMNIST or CIFAR10 **commandline configurable**
batch_size = 64 # Used by torch.utils.data.DataLoader for mini-batching for training

### Model parameters
# gamma is the parameter for fractional posterior: we run with 1, 0.9, 0.5 and 0.1
# gamma = 1.0 is ELBO
# if gamma > 1.0, then we do beta-VAE. The theory class must be VFAE in this case
gamma = 0.5            # **commandline configurable**
theoryclass = "VFAE"   # VFAE, VFAESI, VFBAESI **commandline configurable**

#### Encoder and Decoder of VAE
q = 2 # dimension of latent variable $z$, so that we can plot and see it, from (Ruthotto and Haber, 2021) **commandline configurable**
width_enc = 32 # from (Ruthotto and Haber, 2021)
width_dec = 32 # from (Ruthotto and Haber, 2021)

##### Semi-implicit distribution#, see Yin and Zhou, ICML 2018.
implicit_dim = [15, 10, 5]    # implicit noise dimensions. We make it smaller than because we want ...
implicit_hidden = [28, 14, 2] # the implicit hidden output dimension to be eventually 2 dimensions, so that we can plot and see it

### Inference parameters
#### - The first number of is the number of samples for which data points are generated
#### - For stochastic encoder, the second number is the number of samples from the implicit distribution, generating the first.
#### - For Bayes+Fractional Posterior, the third number is the number of subsamples from the implicit distribution for cross evaluations
nMCMCtrain= (100, 10, 5) # For training
nMCMCval = (1024, 32, 16)  # For testing/validation

### Training parameters
num_epochs = 1              # number of training epochs **commandline configurable**
learning_rate = 1e-3         # input to Adam optimiser
learning_weight_decay = 1e-5 # input to Adam optimiser

### These are not so important
gpudevice = "cuda"      # **commandline configurable**
retrain = False         # **commandline configurable**
dovalidate = False      # **commandline configurable**
dodiagnoistics = False  # **commandline configurable**
generatefromprior = 0   # **commandline configurable**
savefreq = 20           # **commandline configurable**
base_seed = 42
run = -1                # **commandline configurable**


## System setup
- to run from command line, convert this notebook with jupyter nbconvert filename.ipynb --to python
- to run from Google colab, download and copy to your drive

In [ ]:
### Check Environment, so that we can run on Google Colab, Kaggle and Amazon SageMaker Studio Lab
import sys, os

if 'google.colab' in sys.modules:
    print("Colab")
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/')
    dgm_dir = '/content/drive/MyDrive/vlfp/'
    if not os.path.exists(dgm_dir):
        raise Exception("Code does not exists. Download and copy to your drive under the directory vlfp")
    else:
        print("Code already exists")
    sys.path.append(os.path.dirname(dgm_dir)) # Allow our own modules
    os.chdir('/content/drive/MyDrive/vlfp/notebooks/')

    # sensible settings within (free) collab to override key_parameters
    is_budget = True
    is_interactive = lambda: False # True or False?

    # An opportunity to set the commandline arguments in Colab
    if not is_interactive():
      import shlex
      colab_argString = "--device cuda --theoryclass VFAE --data CIFAR10 --q 8 --epochs 20 --gamma 0.001 --skiptrain --skipvalidate --savefreq 5 --run 0 --generate 10" # see vae-expt.sh in the expt directory. ~12hrs on kaggle
      sys.argv = [""] + shlex.split(colab_argString)

elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    print("Kaggle")
    if not os.path.exists('/kaggle/input/vlfp-modules'):
        raise Exception("Code does not exists. Add modules as a dataset named vlfp-modules")
    else:
        print("Code already exists")
    sys.path.append('/kaggle/input/vlfp-modules') # Allow our own modules

    # prepare data
    import shutil
    if not os.path.exists('/kaggle/working/data'):
        os.symlink('/kaggle/input/vlfp-datasets', '/kaggle/working/data')
        #shutil.copytree(os.path.dirname(dgm_dir) + '/data', '/kaggle/working/data') # Use this if you intend to modify the data

    if not os.path.exists('/kaggle/working/results'):
        src_dir = '/kaggle/input/models'
        dest_dir = '/kaggle/working/results'
        if os.path.exists(src_dir):
            shutil.copytree(src_dir, dest_dir) # copy the model directory.
        else:
            os.mkdir(dest_dir)

    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Localhost') == 'Interactive':
        is_interactive = lambda: True
    else:
        is_interactive = lambda: False
        # An opportunity to set the commandline arguments in kaggle
        import shlex
        kaggle_argString = "--device cuda --theoryclass VFAE --data FMNIST --q 4 --epochs 500 --gamma 0.001 --skiptrain --skipvalidate --trainonly --savefreq 20 --run 0" # see vae-expt.sh in the expt directory. ~12hrs on kaggle
        sys.argv = [""] + shlex.split(kaggle_argString)

    is_budget = True
else:
    print("Generic")
    import __main__ as main
    sys.path.append(os.path.abspath('..')) # Allow our own modules
    is_interactive = lambda: not hasattr(main, '__file__')
    is_budget = False

In [ ]:
### Perhaps Load the parameters from commandline
if not is_interactive():
    # we have a chance to change the fixed parameters
    print("Using commandline arguments: %s" % (sys.argv));

    from argparse import ArgumentParser
    argparser = ArgumentParser()
    argparser.add_argument("--data", type=str, default = datasetstr)
    argparser.add_argument("--gamma", type=float)
    argparser.add_argument("--epochs", type=int)
    argparser.add_argument("--theoryclass", type=str, choices=("VFAE", "VFAESI", "VFBAESI", "VFBAESI2"))
    argparser.add_argument("--device", type=str)
    argparser.add_argument("--skiptrain", action="store_true")
    argparser.add_argument("--skipvalidate", action="store_true")
    argparser.add_argument("--trainonly", action = "store_true")
    argparser.add_argument("--generate", type=int, default = generatefromprior)
    argparser.add_argument("--run", type=int, default = run)
    argparser.add_argument("--q", type=int, default = q)
    argparser.add_argument("--savefreq", type=int, default = savefreq)
    args = argparser.parse_args()

    gamma = args.gamma
    q = args.q
    num_epochs = args.epochs
    theoryclass = args.theoryclass
    gpudevice = args.device
    retrain = False if args.skiptrain else True # this would be the default
    dovalidate = False if args.skipvalidate else True # this would be the default
    dodiagnoistics = False if args.trainonly else True # this would be the default
    generatefromprior = args.generate
    datasetstr = args.data
    run = args.run
    savefreq = args.savefreq
elif is_budget: # Or if we are poor
    batch_size = 2048 if batch_size > 2048 else batch_size
    num_epochs = 50 if num_epochs > 50 else num_epochs
    gpudevice = "cuda:0"

In [ ]:
### Make sure that we can indeed run.
import torch
device = gpudevice if torch.cuda.is_available() else "cpu"

if device == 'cpu':
    if not 'global_cpu_count' in globals():
        global_cpu_count = os.cpu_count()
        torch.set_num_interop_threads(global_cpu_count)
        torch.set_num_threads(global_cpu_count)
    print("cpu x", global_cpu_count)
    print("MKL ", torch.backends.mkl.is_available()) # see https://anaconda.org/conda-forge/mkl
else:
    print(device)

device = torch.device(device)

In [ ]:
## Set seed
import random
import numpy as np

def set_seed(seed):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if run >= 0: # we may want to random run sometimes just to test things out
    # Set the seed for reproducibility
    unique_seed = base_seed + run
    set_seed(unique_seed)
    print("Seed=", unique_seed)
else:
    print("Warning: No seed set")
    unique_seed = -1

In [ ]:
if gamma==1.0:
    print("This is ELBO")
elif gamma > 1.0:
    if theoryclass != "VFAE":
        raise Exception("Invalid combination of theoryclass and gamma")
    print("This is beta-VAE")

In [ ]:
out_file = ("./results/%s-%s-q-%d-e-%d-d-%d-g-%012.5f-n-%d-b-%d-s-%d") % (theoryclass, datasetstr.lower(), q, width_enc, width_dec, gamma, nMCMCtrain[0], batch_size, unique_seed)
if not os.path.exists(os.path.dirname(out_file)):
    os.mkdir(os.path.dirname(out_file))

In [ ]:
### print Key parameters for logging
print("theoryclass=%s, batch=%d, gamma=%012.5f, q=%d, enc=%d, dec=%d, epochs=%d, MCMCtrain=(%d %d %d)" % ((theoryclass, batch_size, gamma, q, width_enc, width_dec, num_epochs) + nMCMCtrain))
print("File=%s" % (out_file))

## Prepare Image Data

https://pytorch.org/vision/stable/transforms.html#supported-input-types-and-conventions
- Tensor image are expected to be of shape (C, H, W), where C is the number of channels, and H and W refer to height and width. Most transforms support batched tensor input. A batch of Tensor images is a tensor of shape (N, C, H, W), where N is a number of images in the batch. The v2 transforms generally accept an arbitrary number of leading dimensions (..., C, H, W) and can handle batched images or batched videos.

For data sets, see https://pytorch.org/vision/stable/datasets.html

In [ ]:
### Nice plotting
import matplotlib.pyplot as plt

### Image stuff
import torchvision
from torch.utils.data import DataLoader

if datasetstr == "MNIST":
    from torchvision.datasets import MNIST
elif datasetstr == "FMNIST":
    from torchvision.datasets import FashionMNIST as MNIST
elif datasetstr == "CIFAR10":
    from torchvision.datasets import CIFAR10 as MNIST
else:
    raise Exception("Data Set not supported")

img_transform = torchvision.transforms.ToTensor()

train_dataset = MNIST(root='./data/', download=True, train=True, transform=img_transform)
test_dataset = MNIST(root='./data/', download=True, train=False, transform=img_transform)

print(train_dataset, '\n\n', test_dataset)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
x,_ = next(iter(train_dataloader))
numchannels, imagesz = x.shape[1], x.shape[2]
if( imagesz != x.shape[3]):
    raise Exception("Images are not square")

cmap = 'brg' if numchannels == 3 else 'Greys'
def make_grid(x, ncol):
  if numchannels == 3:
    return torchvision.utils.make_grid(x, ncol, padding=1,pad_value=1.0).permute(1,2,0)
  else:
    return torchvision.utils.make_grid(x, ncol, padding=1,pad_value=1.0)[0]

def make_img(x):
  if numchannels == 3:
    return x.permute(1,2,0)
  else:
    return x[0]

In [ ]:
fig = plt.Figure()
if hasattr(fig, "show"):
   plt.imshow(make_grid(x,16), vmin=0, vmax=1, cmap=cmap)
   plt.axis("off")
   plt.margins(0, 0)
   plt.show()

## Setup Model

In [ ]:
from nnarch.vaenn import *
from inf.vfbae import *

g = Generator(imagesz, numchannels, width_dec,q)

if theoryclass == "VFAE":
    e = Encoder(imagesz, numchannels, width_enc,q)
    VAE = VFAE
    # Just need one number
    nMCMCtrain = nMCMCtrain[0]
    nMCMCval = nMCMCval[0]
elif theoryclass == "VFAESI":
    e = EncoderSI(imagesz, numchannels, width_enc, q, implicit_dim, implicit_hidden)
    VAE = VFAESI
elif theoryclass == "VFBAESI":
    e = EncoderSIB(imagesz, numchannels, width_enc, q, implicit_dim, implicit_hidden)
    VAE = VFBAESI
elif theoryclass == "VFBAESI2":
    e = EncoderSIB(imagesz, numchannels, width_enc, q, implicit_dim, implicit_hidden)
    VAE = VFBAESI2
else:
    raise ValueError("Unknown theory class " + theoryclass)

if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    e = torch.nn.DataParallel(e)
    g = torch.nn.DataParallel(g)
    print("Using multiple GPUs")

vae = VAE(e, g, gamma).to(device)

## Train the Generator and Approximate Posterior

In [ ]:
def singleEpoch(model, dataloader, nMCMC, optimizer = None):
    num_ex = 0

    stat = None

    for image_batch, _ in dataloader:
        image_batch = image_batch.to(device)
        scores = model.ELBOscores(image_batch, nMCMC)

        if optimizer is not None:
            optimizer.zero_grad()
            scores[0].backward()
            optimizer.step()

        this_num_ex = image_batch.shape[0]
        this_scores = np.array([ scores[0].item(), *scores[1:]]);

        num_ex += this_num_ex;
        if stat is None:
            stat = this_scores * this_num_ex
        else:
            stat += this_scores * this_num_ex

    stat /= num_ex
    return stat

def stat2str(t):
    return tuple(f"{num:0.3f}" for num in t);

### Here, we use ADAM, a stochastic approximation scheme that operates on minibatches.

In [ ]:
from collections import OrderedDict
from scipy.io import savemat

def savept(out_file, vae, his):
    print("Saving")
    his = np.stack(his)
    savemat(("%s.mat") % (out_file), {"his":his})
    if not isinstance(vae.g, nn.DataParallel):
        torch.save(vae.g.state_dict(), ("%s-g.pt") % (out_file))
        torch.save(vae.state_dict(), ("%s.pt") % (out_file))
    else:
        torch.save(vae.g.module.state_dict(), ("%s-g.pt") % (out_file))
        edict = {"e."+ k: v for k, v in vae.e.module.state_dict().items()}
        gdict = {"g."+ k: v for k, v in vae.g.module.state_dict().items()}
        torch.save( OrderedDict(edict | gdict), ("%s.pt") % (out_file))

In [ ]:
from os import path

if retrain == False and path.exists(out_file + ".pt"):
    vae.load_state_dict(torch.load("%s.pt" % (out_file), weights_only=True, map_location=torch.device(device)))
    print("loaded results from %s" % out_file)

    if path.exists(out_file + ".mat"):
        from scipy.io import loadmat
        his_file = loadmat("%s.mat" % (out_file))
        his = his_file['his']
        already_trained_epochs = len(his)
        his = [np.array(h) for h in his]
    else: # assume fully trained model
        print("No history file, assumed fully trained")
        already_trained_epochs = num_epochs
else:
    if out_file is not None:
        import os
        out_dir, fname = os.path.split(out_file)
        if not os.path.exists(out_dir):
            os.makedirs(out_dir)
        print((3*"--" + "out_file: %s" + 3*"--") % (out_file))

    print((3*"--" + "device=%s, q=%d, batch_size=%d, num_epochs=%d, w_enc=%d, w_dec=%d" + 3*"--") % (device, q, batch_size, num_epochs, width_enc, width_dec))
    his = []
    already_trained_epochs = 0


if already_trained_epochs < num_epochs:
    vae.train()
    print("Total number of parameters = ", sum(p.numel() for p in vae.parameters()))

    optimizer = torch.optim.Adam(params=vae.parameters(), lr=learning_rate, weight_decay=learning_weight_decay)
    for epoch in range(already_trained_epochs, num_epochs):
        trainstats = singleEpoch(vae, train_dataloader, nMCMCtrain, optimizer)
        print(("%06d  %s  ") % (epoch + 1, stat2str(trainstats)))
        his = his + [trainstats]
        if out_file is not None and savefreq > 0 and epoch > 0:
            if epoch % savefreq == 0:
                savept(out_file, vae, his)

    if out_file is not None:
        savept(out_file, vae, his)

In [ ]:
class StopExecution(Exception):
    pass

def exit_notebook(): raise StopExecution

if not dodiagnoistics:
    exit_notebook()

## Testing and Evaluation

In [ ]:
vae.eval();

### Print the fit with more MCMC samples and also ELBO

In [ ]:
def printtable(desc, scores):
    s = tuple(f"{num:0.3f}" for num in scores)
    print(desc, s)
    return " & ".join(s)

In [ ]:
if dovalidate:
    print("Validating: theoryclass=%s gamma=%.5f num_epochs=%d seed=%d" % (theoryclass, gamma, num_epochs, unique_seed));
    s = theoryclass + " & " + f"{gamma:0.1f}"
    with torch.no_grad():
        s = s + " & " + printtable("Train: ", -singleEpoch(vae, train_dataloader, nMCMCval));
        s = s + " & " + printtable("Validation: ", -singleEpoch(vae, test_dataloader, nMCMCval));
        if theoryclass == "VFBAESI" or theoryclass == "VFBAESI2":
            other = VFAESI(vae.e.bayes(), vae.g, 1.0)
            scores = -singleEpoch(other, test_dataloader, nMCMCval)
        else:
            vae.gamma = 1.0 # We do ELBO here
            scores = -singleEpoch(vae, test_dataloader, nMCMCval)
            vae.gamma = gamma # Make sure we are the correct setting
        s = s + " & " + printtable("Validation ELBO: ", scores) + " \\\\"
    print(s)
    with open("%s-validate.txt" % out_file, "w") as text_file:
        text_file.write(s)

### Check implicit distribution (only for semi-implicit latents)

In [ ]:
if not hasattr(vae, "ELBO_and_implicit"):
    print("No ELBO_and_implicit")
else:
    from torch.utils.data import SequentialSampler
    dataloader = DataLoader(train_dataset, batch_size=batch_size, sampler=SequentialSampler(train_dataset))
    vae.e.nMCMC = 0 # to do for the rest for evaluation!

    nsamples = 500  #get 500 implicit samples from the first batch (64 images)
    implicits = []
    with torch.no_grad():
        image_batch,_ = next(iter(dataloader))
        image_batch = image_batch.to(device)
        implicit = vae.e.forward_implicit(image_batch, nsamples)
        implicits.append(implicit)
    implicits = torch.cat(implicits,0).cpu().numpy()

    fig=plt.Figure()
    if hasattr(fig, "show"):
        fig, axs = plt.subplots(8, 8)
        for i in range(8):
            for j in range(8):
                toplot = implicits[:, i*8+j, :]
                axs[i,j].axis((-0.01, 1.01, -0.01, 1.01))
                axs[i,j].set_xticks([])
                axs[i,j].set_yticks([])
                axs[i,j].scatter(toplot[:,0],toplot[:,1], s=1)
        plt.show()

    # subsample to 25 images for saving
    implicits = implicits[:, 0:25, :].transpose(1,0,2).reshape(-1, 2)
    indx = np.tile(range(25), (nsamples, 1)).T.reshape(-1,1);
    implicits = np.hstack((indx, implicits))
    np.savetxt("%s-is.csv" % out_file , implicits, fmt="%d,%f,%f")

    # check for all image, one sample per image
    implicits = []
    with torch.no_grad():
        for image_batch, _ in test_dataloader:
            image_batch = image_batch.to(device)
            implicit = vae.e.forward_implicit(image_batch, 0)
            implicits.append(implicit)
    implicits = torch.cat(implicits,0).cpu().numpy()

    fig=plt.Figure()
    if hasattr(fig, "show"):
        plt.scatter(implicits[:,0], implicits[:,1])
    plt.show()

    # subsample to have 5,000 rows only, since we can only plot 5,000 in TikZ anyway
    np.savetxt("%s-ia.csv" % out_file , implicits[::2], fmt="%f,%f")

    id=47 # This is for semi-implicit ELBO, gamma=1.0
    with torch.no_grad():
        image_batch, label = train_dataloader.dataset[id]
        image_batch = image_batch.to(device).unsqueeze(0)

        us = []; elbos = []
        for i in range(0,500):
            nelbo, u = vae.ELBO_and_implicit(image_batch, (1000, 1, 500))
            nelbo = nelbo.cpu()
            u = u.squeeze(0,1).cpu().numpy()
            us.append(u); elbos.append(-nelbo)
        us = np.array(us); elbos = np.array(elbos)

        fig=plt.Figure()
        toplot = implicit.squeeze(1)
        plt.title("%d" % (id))
        plt.axis((-0.01, 1.01, -0.01, 1.01))
        plt.xticks([])
        plt.yticks([])
        plt.scatter(us[:,0],us[:,1], c=elbos, cmap="Greys")
        plt.show()


### Show Samples
- Test samples
- 2-D interpolation in latent space

In [ ]:
ncol = 9
x0, _ = next(iter(test_dataloader))
x1, _ = next(iter(test_dataloader))
x = torch.concatenate( (x0, x1), dim=0 )[0:(ncol*ncol), :, :, :]

x_samples = make_grid(x.cpu().detach(), ncol)
plt.imsave("%s-testset-9x9.png" % out_file, x_samples.cpu().numpy(), vmin=0, vmax=1, cmap=cmap);

fig = plt.Figure(frameon=False)
if hasattr(fig, "show"):
    plt.subplot(1,2,1)
    plt.imshow(x_samples, vmin=0, vmax=1, cmap=cmap)
    plt.axis("off")
    plt.margins(0, 0)

if q == 2: # only do this for 2-D
    from statistics import NormalDist
    with torch.no_grad():
        pam = torch.from_numpy(np.array([NormalDist().inv_cdf(x) for x in np.linspace(0.1, 0.9, ncol)])).float().to(device)
        mxx, myy = torch.meshgrid(pam, pam, indexing="ij")
        mxx = mxx.flatten().unsqueeze(1)
        myy = myy.flatten().unsqueeze(1)
        mzz = torch.cat( (mxx, myy), dim=1)
        x_vae = vae.mean_x(mzz) #  Since we are only testing the decoder, the encoder (whichever posterior) doesn't matter

    x_samples = make_grid(x_vae.cpu().detach(), ncol)
    plt.imsave("%s-2d.png" % out_file, x_samples.cpu().numpy(), vmin=0, vmax=1, cmap=cmap);

    if hasattr(fig, "show"):
        plt.subplot(1,2,2)
        plt.imshow(x_samples, vmin=0, vmax=1, cmap=cmap)
        plt.axis("off")
        plt.margins(0,0)

if hasattr(fig, "show"):
   plt.show()

In [ ]:
generated = 0
while generated < generatefromprior:
    left = generatefromprior - generated
    n = batch_size if left > batch_size else left
    z = torch.randn((n,q)).to(device)
    with torch.no_grad():
        x_vae = vae.mean_x(z).cpu().detach()
    for i in range(n):
      plt.imsave("%s-sample-%05d.png" % (out_file, generated+i), make_img(x_vae[i,:,:,:]).numpy(), vmin=0, vmax=1, cmap=cmap)
    generated = generated + n

### Prepare for subsequent steps

In [ ]:
mu_val = []
label_val = []
loglike_val = []
logprior_val = []
zs_val = []
vae.e.nMCMC=0
for x, label_batch in test_dataloader:
        with torch.no_grad():
            x = x.to(device)
            s = x.shape[1]
            mu, logvar = vae.encode(x)
            z, _ = vae.sample_ezx(mu, logvar)
            gz = vae.g(z)

            log_pzx, log_like, log_prior = vae.log_prob_pzx(z, x, gz)

            zsamples, _ = vae.sample_ezx(mu, logvar, 10) # this is missing from the Tutorial's code
            zsamples = zsamples.view( (-1, zsamples.shape[-1]))

            mu_val.append(mu)
            label_val.append(label_batch)
            loglike_val.append(log_like)
            logprior_val.append(log_prior)

            zs_val.append(zsamples);

mu_val = torch.cat(mu_val,0).cpu().numpy()
label_val = torch.cat(label_val,0).cpu().numpy()
loglike_val = torch.cat(loglike_val,0).cpu().numpy()
logprior_val = torch.cat(logprior_val,0).cpu().numpy()
zs_val = torch.cat(zs_val,0).cpu().numpy()

### Show Posterior Approximation

In [ ]:
## 5) show joint probability p(z,x) and approximation e(z|x) for some examples
if q==2:
  imin = np.argsort(loglike_val)[:1]
  imax = np.argsort(loglike_val)[-1:]

  with torch.no_grad():
      vae.eval()
      z1 = torch.linspace(-5.5, 5.5, 100)
      z2 = torch.linspace(-5.5, 5.5, 100)
      zg = torch.meshgrid(z1, z2, indexing="ij")
      z = torch.cat((zg[0].reshape(-1, 1), zg[1].reshape(-1, 1)), 1).to(device)

      # reconstruct images from the latent vectors
      gz = vae.g(z)

      # compute posterior
  cnt=0
  for ind in np.hstack((imin,imax)):
      vae.eval()
      print("ind=%d" % ind)
      x, _ = test_dataloader.dataset[ind]
      x = x.to(device).unsqueeze(0)
      zt = vae.encode(x)[0].detach()
      gzt = vae.g(zt)
      log_pzx, log_like, log_prior = vae.log_prob_pzx(z, torch.cat(len(z1)*len(z2)*[x]), gz)
      log_ezx = vae.log_prob_ezx(z,torch.cat(len(z1)*len(z2)*[x]))
      zm = z[np.argmax(log_pzx.cpu().numpy()), :]
      gzm = vae.g(zm.unsqueeze(0))

      fig=plt.Figure()
      if hasattr(fig, "show"):
          plt.subplot(cnt+1,5,1+cnt*5)
          img = log_pzx.reshape(len(z1),len(z2))
          plt.contour(z1,z2,img.cpu().numpy(),40,linewidths=2)
          plt.plot(zm[1].cpu(),zm[0].cpu(),"bs")
          plt.plot(zt[0,1].cpu(),zt[0,0].cpu(),"or")
          plt.xlabel(r"$\mathbf{z}_1$", labelpad=-20)
          plt.ylabel(r"$\mathbf{z}_2$", labelpad=-30)
          plt.axis('square')
          plt.axis((z1[0], z1[-1], z2[0], z2[-1]))
          plt.xticks((z1[0], z1[-1]))
          plt.yticks((z2[0], z2[-1]))
          if cnt==0:
              plt.title("log_pzx")

          plt.subplot(cnt+1,5,2+cnt*5)
          img = log_ezx.reshape(100,100)
          plt.contour(z1,z2,img.detach().cpu().numpy(),30,linewidths=2)
          plt.plot(zt[0,1].cpu(),zt[0,0].cpu(),"or")
          plt.xlabel(r"$\mathbf{z}_1$", labelpad=-20)
          plt.ylabel(r"$\mathbf{z}_2$", labelpad=-30)
          plt.axis('square')
          plt.axis((z1[0], z1[-1], z2[0], z2[-1]))
          plt.xticks((z1[0], z1[-1]))
          plt.yticks((z2[0], z2[-1]))
          if cnt==0:
              plt.title("log_ezx")

          plt.subplot(cnt+1,5,3+cnt*5)
          plt.imshow( make_img(x.cpu().squeeze(0)), vmin=0, vmax=1, cmap=cmap)
          # plt.axis('square')
          plt.axis('off')
          plt.margins(0, 0)
          if cnt==0:
              plt.title("true image")

          plt.subplot(cnt+1,5,4+cnt*5)
          plt.imshow( make_img(gzt[0].detach().cpu().squeeze(0)), vmin=0, vmax=1, cmap=cmap)
          # plt.axis('square')
          plt.axis('off')
          plt.margins(0, 0)
          if cnt==0:
              plt.title("MAP e_zx")

          plt.subplot(cnt+1,5,5+cnt*5)
          plt.imshow( make_img(gzm[0].detach().cpu().squeeze(0)), vmin=0, vmax=1, cmap=cmap)
          plt.axis('off')
          plt.margins(0, 0)
          if cnt==0:
              plt.title("MAP p_zx")
      cnt+=1

### Show Latent Space Structure

In [ ]:
fig=plt.Figure()
if hasattr(fig, "show") and q==2:
    plt.subplot(1,2,1)
    plt.scatter(mu_val[:,0],mu_val[:,1],c=label_val)
    #plt.axis("square")
    #plt.axis((-3.5, 3.5, -3.5, 3.5))
    plt.xticks((-3.5, 3.5))
    plt.yticks((-3.5, 3.5))
    plt.xlabel(r"$\mathbf{z}_1$", labelpad=-20)
    plt.ylabel(r"$\mathbf{z}_2$", labelpad=-30)

    plt.subplot(1,2,2)
    plt.hist2d(zs_val[:,0],zs_val[:,1],bins=256,range=[[-3.5,3.5],[-3.5,3.5]])
    plt.axis("square")
    plt.axis((-3.5, 3.5, -3.5, 3.5))
    plt.xticks((-3.5, 3.5))
    plt.yticks((-3.5, 3.5))
    plt.xlabel(r"$\mathbf{z}_1$", labelpad=-20)
    plt.ylabel(r"$\mathbf{z}_2$", labelpad=-30)

In [ ]:
# save for plotting in LaTeX
# subsample to have 5,000 rows only, since we can only plot 5,000 in TikZ anyway
if q == 2:
    np.savetxt("%s-mu.csv" % out_file , (np.concatenate((mu_val, np.expand_dims(label_val,1)), axis=1))[::2], fmt="%f,%f,%d")
    np.savetxt("%s-zs.csv" % out_file , zs_val[::20], fmt="%f,%f")

In [ ]:
if ("KAGGLE_KERNEL_RUN_TYPE" in os.environ) and is_interactive():
    from IPython.display import FileLink
    %cd /kaggle/working
    !tar -zcf results.tar.gz results
    FileLink(r'results.tar.gz') # You may need to copy and run this in a separate cell.